# Train the Finance Analyzer on Google Colab

Select **Runtime → Change runtime type → GPU**. Upload the source/data ZIP produced with `python scripts/package_project.py --dataset`. LayoutLMv3 is CC-BY-NC-SA-4.0; this notebook is for academic/noncommercial project use. Training on synthetic data does not establish real-world accuracy.

In [ ]:
from google.colab import files
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith(".zip"))
import zipfile, pathlib, os
project = pathlib.Path("/content/finance-analyzer")
project.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as bundle:
    root = project.resolve()
    for name in bundle.namelist():
        if not (root / name).resolve().is_relative_to(root):
            raise ValueError("Unsafe archive path")
    bundle.extractall(project)
os.chdir(project)

In [ ]:
import subprocess
subprocess.run('apt-get -qq update && apt-get -qq install -y tesseract-ocr tesseract-ocr-hin poppler-utils fonts-dejavu-core',shell=True,check=True)
subprocess.run('pip -q install transformers==4.42.4 accelerate==0.32.1 seqeval==1.2.2 Pillow==10.4.0 pytesseract==0.3.10 numpy==1.26.4',shell=True,check=True)
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU; select a GPU runtime")
assert torch.cuda.is_available(), "Enable a GPU runtime before full training"

In [ ]:
import subprocess
from pathlib import Path
if not Path("data/training_indian_invoices/manifest.json").exists():
    subprocess.run('python data/build_training_dataset.py --n 1000 --out data/training_indian_invoices',shell=True,check=True)


## Full training

This uses all 700 training images, 150 validation images, and 150 held-out test images. Validation selects the checkpoint; test metrics are reported after training. If a modern Colab image has incompatible preinstalled libraries, use a fresh runtime and rerun setup.

In [ ]:
import subprocess
subprocess.run('python notebooks/finetune_layoutlmv3.py --data_dir data/training_indian_invoices --output_dir models/layoutlmv3-indian-invoices --epochs 3 --batch_size 1 --gradient_checkpointing',shell=True,check=True)


In [ ]:
import json
for split in ["validation", "test"]:
    print(split, json.loads(Path(f"models/layoutlmv3-indian-invoices/{split}_results.json").read_text()))

In [ ]:
import shutil
shutil.make_archive("trained-layoutlmv3", "zip", "models/layoutlmv3-indian-invoices")
files.download("trained-layoutlmv3.zip")

Extract the downloaded checkpoint into `models/layoutlmv3-indian-invoices` on your laptop. Restart with `scripts/start.ps1 -Mode token`. Keep the model files and the JSON metrics together for your report.